# NL_SQL autotune — Kaggle runner (plan_autotune S2–S4)

One notebook, two modes (set `MODE` below, push a new version per run):
- **train** — QLoRA via `train_qlora.py` from the attached dataset; adapter lands in `/kaggle/working/adapter.zip`. Session cut mid-run? Attach this kernel's previous output as `RESUME_INPUT` and re-push: checkpoints are copied in and training resumes.
- **serve** — vLLM (base model, plus LoRA adapter when `ADAPTER_DIR` exists) + cloudflared quick tunnel. The public URL is printed and written to `/kaggle/working/tunnel_url.txt`; put it into `NL_SQL_LOCAL_LLM_BASE_URL` on the Windows side (add `/v1`).

Needs: GPU (T4x2 preferred) + Internet ON in notebook settings.

In [ ]:
MODE = "train"  # "train" | "serve"

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATA_DIR = "/kaggle/input/nlsql-autotune-data"  # train.jsonl, val.jsonl, train_qlora.py
OUT = "/kaggle/working/qlora_out"

# train mode
EPOCHS = 1.0
MAX_SEQ = 4096
SMOKE_STEPS = 5  # >0: run this many steps end-to-end first, then the real run (0 = straight in)
RESUME_INPUT = ""  # e.g. "/kaggle/input/nlsql-autotune-prev" (previous run's output) or ""

# serve mode
ADAPTER_DIR = "/kaggle/input/nlsql-adapter/adapter"  # absent path => serve plain base model
SERVE_HOURS = 5.0
PORT = 8000

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

if MODE == "train":
    tree = subprocess.run(
        ["find", "/kaggle/input", "-maxdepth", "4"], capture_output=True, text=True
    ).stdout
    print(tree, flush=True)
    hits = sorted(Path("/kaggle/input").rglob("train_qlora.py"))
    if not hits:
        raise RuntimeError("train_qlora.py not found anywhere under /kaggle/input")
    data_dir = hits[0].parent
    print(f"data_dir resolved: {data_dir}", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], check=True)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "datasets>=3.0",
            "trl>=0.12",
            "peft>=0.13",
            "bitsandbytes>=0.44",
            "accelerate>=1.0",
        ],
        check=True,
    )

    def train_cmd(out, extra):
        return [
            sys.executable,
            str(data_dir / "train_qlora.py"),
            "--model",
            BASE_MODEL,
            "--train",
            str(data_dir / "train.jsonl"),
            "--val",
            str(data_dir / "val.jsonl"),
            "--out",
            out,
            "--max-seq-len",
            str(MAX_SEQ),
            "--batch-size",
            "1",
            "--grad-accum",
            "16",
            "--no-merge",
            *extra,
        ]

    if SMOKE_STEPS:
        # Both earlier sessions died in their first minutes on a trl kwarg rename,
        # and each failure cost a day because nobody was watching. Prove the whole
        # path -- config, trainer, a real step, evaluate, adapter save -- on a few
        # steps before committing the 6-12h run.
        smoke_out = "/kaggle/working/smoke_out"
        smoke = train_cmd(smoke_out, ["--epochs", "1", "--max-steps", str(SMOKE_STEPS)])
        print(" ".join(smoke), flush=True)
        subprocess.run(smoke, check=True)
        if not Path(smoke_out, "adapter", "adapter_config.json").exists():
            raise RuntimeError("smoke run finished but saved no adapter")
        shutil.rmtree(smoke_out)
        print("SMOKE OK -- starting the real run", flush=True)

    ckpt_dst = Path(OUT) / "checkpoints"
    prev = Path(RESUME_INPUT) / "qlora_out" / "checkpoints" if RESUME_INPUT else None
    if prev and prev.is_dir() and not ckpt_dst.exists():
        shutil.copytree(prev, ckpt_dst)
        print(f"resume: copied checkpoints from {prev}")
    cmd = train_cmd(OUT, ["--epochs", str(EPOCHS), "--resume"])
    print(" ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    shutil.make_archive("/kaggle/working/adapter", "zip", OUT, "adapter")
    print("DONE: /kaggle/working/adapter.zip")

In [ ]:
import re
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

if MODE == "serve":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm>=0.6.3"], check=True)
    cf = "/kaggle/working/cloudflared"
    if not Path(cf).exists():
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            cf,
        )
        subprocess.run(["chmod", "+x", cf], check=True)
    import torch

    tp = 2 if torch.cuda.device_count() >= 2 else 1
    with_lora = Path(ADAPTER_DIR).is_dir()
    vllm_cmd = [
        "vllm",
        "serve",
        BASE_MODEL,
        "--port",
        str(PORT),
        "--dtype",
        "float16",
        "--gpu-memory-utilization",
        "0.92",
        "--max-model-len",
        "12288" if tp == 2 else "8192",
        "--tensor-parallel-size",
        str(tp),
    ]
    if with_lora:
        vllm_cmd += [
            "--enable-lora",
            "--max-lora-rank",
            "16",
            "--lora-modules",
            f"sqltuned={ADAPTER_DIR}",
        ]
    print(" ".join(vllm_cmd), flush=True)
    with open("/kaggle/working/vllm.log", "w") as vllm_log:
        vllm_proc = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT)
    for _ in range(240):  # up to 40 min: first run downloads ~15GB of weights
        time.sleep(10)
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=5)
            break
        except Exception:
            if vllm_proc.poll() is not None:
                print(Path("/kaggle/working/vllm.log").read_text()[-3000:])
                raise RuntimeError("vllm died during startup") from None
    else:
        raise RuntimeError("vllm never became ready")
    print("vllm ready", flush=True)
    tun = subprocess.Popen(
        [cf, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    url = None
    assert tun.stdout is not None
    for line in tun.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0)
            break
    Path("/kaggle/working/tunnel_url.txt").write_text(url or "NONE")
    model_name = "sqltuned" if with_lora else BASE_MODEL
    print(
        f"\n{'=' * 60}\nTUNNEL: {url}\nbase_url for .env: {url}/v1\n--sql-model {model_name}\n{'=' * 60}\n",
        flush=True,
    )
    deadline = time.time() + SERVE_HOURS * 3600
    while time.time() < deadline and vllm_proc.poll() is None:
        time.sleep(300)
        print(f"alive, url={url}, {int((deadline - time.time()) / 60)} min left", flush=True)
    print("serve window over")